[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/concept_notebooks/How_Weights_Are_Learned.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/concept_notebooks/How_Weights_Are_Learned.ipynb)

# How Machines Learn Feature Weights
## From Random Numbers to Meaningful Rules

**Dataset**: Synthetic (we build it ourselves so we know the true answer)
**Core question**: *How does a model start from random weights and learn the right ones?
Is the weight determined by the feature's distribution?*

---

### What this notebook answers

1. What is an observation? What is a feature? What is a target?
2. What does "a rule" actually mean as a formula?
3. How does the model start from random weights and update them?
4. Does the feature's distribution (normal, skewed, binary) determine its weight?
5. How does feature scaling affect the learning process?
6. How do we read what the final weights are telling us?

## Section 1 — The Dataset: Rows, Columns, Features, Target

### Every dataset is a table of observations

Each **row** is one observation — one real-world thing that was measured.
Each **column** is one feature — one thing we measured about it.
One special column is the **target** — what we want to predict.

```
observation  │  age  │  income ($)  │  score (0–100)  │  is_new  │  TARGET: spend ($)
─────────────┼───────┼──────────────┼─────────────────┼──────────┼────────────────────
person 1     │   28  │   45,000     │      72         │    1     │      142
person 2     │   45  │  110,000     │      88         │    0     │      267
person 3     │   33  │   62,000     │      61         │    1     │      189
...
```

### Four features — four different distributions

We build a synthetic dataset with **four features** that have deliberately different distributions:

| Feature | Distribution | Range | Reason for choosing it |
|---|---|---|---|
| `age` | Normal (bell curve) | ~10–70 | Many real-world measurements |
| `income` | Log-normal (right-skewed) | ~$20k–$300k | Money is always right-skewed |
| `score` | Uniform (flat) | 0–100 | A rating scale with no bias |
| `is_new` | Binary (Bernoulli) | 0 or 1 | A yes/no flag |

### The true rule (only we know this — the model must discover it)

```
spend = 0.8 × age  +  0.0003 × income  +  1.5 × score  −  30 × is_new  +  50  +  noise
```

The model will start with **random weights** and must learn these values from data alone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

N = 400

# ── Four features with different distributions ────────────────────────────────
age    = np.random.normal(35, 12, N).clip(18, 70)        # Normal
income = np.random.lognormal(10.8, 0.5, N)               # Log-normal (skewed)
score  = np.random.uniform(0, 100, N)                     # Uniform
is_new = np.random.binomial(1, 0.35, N).astype(float)    # Binary

# ── The TRUE weights (the model must discover these) ─────────────────────────
TRUE_W = {'age': 0.8, 'income': 0.0003, 'score': 1.5, 'is_new': -30.0}
TRUE_B = 50.0

noise = np.random.normal(0, 8, N)
spend = (TRUE_W['age']    * age    +
         TRUE_W['income'] * income +
         TRUE_W['score']  * score  +
         TRUE_W['is_new'] * is_new +
         TRUE_B + noise)

df = pd.DataFrame({'age': age, 'income': income, 'score': score,
                   'is_new': is_new, 'spend': spend})

print("Dataset shape:", df.shape)
print()
print("Feature statistics:")
print(df.describe().round(1).to_string())
print()
print("True weights the model must learn:")
for feat, w in TRUE_W.items():
    print(f"  {feat:8s}: {w:+.4f}")

In [ ]:
# Visualise the four feature distributions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
configs = [
    ('age',    'steelblue',  'Normal distribution\n(bell curve, symmetric)'),
    ('income', '#FFA726',    'Log-normal distribution\n(right-skewed, long tail)'),
    ('score',  '#26A69A',    'Uniform distribution\n(flat, equally likely)'),
    ('is_new', '#AB47BC',    'Binary distribution\n(only 0 or 1)'),
]
for ax, (col, color, title) in zip(axes, configs):
    if col == 'is_new':
        counts = df[col].value_counts().sort_index()
        ax.bar(counts.index, counts.values, color=color, width=0.4)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['0 (existing)', '1 (new)'])
    else:
        ax.hist(df[col], bins=25, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
plt.suptitle('Four Features — Each Has a Different Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Section 2 — The Rule: Random Weights vs True Weights

### The linear rule

The model's prediction formula is:
```
spend_predicted = w_age × age  +  w_income × income  +  w_score × score  +  w_is_new × is_new  +  b
```

- `w_age`, `w_income`, etc. are the **weights** — what we are learning
- `b` is the **bias** — a constant baseline (like a starting price)

At the start of training, **all weights are set randomly** (or to zero).
The model has no idea yet which features matter or in which direction.

### Do starting weights need to follow a normal distribution?

No.
A normal distribution is just **one common way** to generate starting weights.
For simple linear regression, starting from zeros or small random values usually works.
For larger neural networks, initialization matters more, so people use carefully designed methods such as Xavier or He initialization.

The important idea is not "normal distribution" by itself.
The important idea is that the starting weights should let learning begin in a stable way.

### What happens with random weights?

The model makes predictions using whatever random weights it has.
The predictions are terrible — but that is fine. The error tells us which direction to fix each weight.

In [ ]:
FEATURES = ['age', 'income', 'score', 'is_new']
X = df[FEATURES].values
y = df['spend'].values

# Random initial weights
np.random.seed(7)
w_random = np.random.randn(4)
b_random = np.random.randn()

# True weights
w_true = np.array([TRUE_W[f] for f in FEATURES])
b_true = TRUE_B

# Predictions
y_random = X @ w_random + b_random
y_true_pred = X @ w_true  + b_true  # perfect (only noise remains)

mse_random = np.mean((y_random - y)**2)
mse_true   = np.mean((y_true_pred - y)**2)

print("Random weights:")
for feat, w in zip(FEATURES, w_random):
    print(f"  {feat:8s}: {w:+.4f}")
print(f"  bias    : {b_random:+.4f}")
print(f"  MSE with random weights : {mse_random:,.1f}")
print()
print("True weights:")
for feat, w in zip(FEATURES, w_true):
    print(f"  {feat:8s}: {w:+.4f}")
print(f"  bias    : {b_true:+.4f}")
print(f"  MSE with true weights   : {mse_true:,.1f}  (only noise remains)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(y, y_random, alpha=0.3, s=15, color='#EF5350')
lim1 = (min(y.min(), y_random.min()) - 10, max(y.max(), y_random.max()) + 10)
ax1.plot(lim1, lim1, 'k--', lw=1.5, label='Perfect prediction')
ax1.set_xlabel('Actual spend ($)', fontsize=11)
ax1.set_ylabel('Predicted spend ($)', fontsize=11)
ax1.set_title(f'Random weights\nMSE = {mse_random:,.0f}   Predictions are meaningless', fontsize=11)
ax1.legend()

ax2.scatter(y, y_true_pred, alpha=0.3, s=15, color='#26A69A')
ax2.plot([y.min()-5, y.max()+5], [y.min()-5, y.max()+5], 'k--', lw=1.5, label='Perfect prediction')
ax2.set_xlabel('Actual spend ($)', fontsize=11)
ax2.set_ylabel('Predicted spend ($)', fontsize=11)
ax2.set_title(f'True weights\nMSE = {mse_true:,.0f}   Almost perfect (only noise remains)', fontsize=11)
ax2.legend()

plt.suptitle('Random Weights vs True Weights — Same Data, Very Different Results', fontsize=12)
plt.tight_layout()
plt.show()

### Before the gradient: what is MSE and why do we need it?

A learning algorithm needs two things:

1. a way to measure **how wrong** the model is
2. a way to decide **how to change the weights**

The first job is done by **MSE**: Mean Squared Error.

```python
MSE = (1/N) × Σ (predicted - actual)^2
```

For each observation:
- compute the prediction error
- square it so negative and positive mistakes do not cancel
- average across all observations

So MSE answers the question:

**"Overall, how bad are the model's predictions?"**

A large MSE means the predictions are far from the true values.
A small MSE means the predictions are close.

The second job is done by the **gradient**.
The gradient tells each weight which direction to move in order to reduce MSE.

So the relationship is:
- **MSE** tells us how much error the model has
- **gradient** tells us how to reduce that error

You can think of it this way:
- MSE is the score of the model's mistakes
- gradient is the correction signal for the weights

In [ ]:
# Tiny numeric example: MSE first, then gradient
x_small = np.array([2.0, 3.0, 4.0])
y_small = np.array([4.0, 6.0, 8.0])
w_small = 1.0

pred_small = w_small * x_small
errors_small = pred_small - y_small
mse_small = np.mean(errors_small ** 2)
grad_small = np.mean(errors_small * x_small)

small_df = pd.DataFrame({
    'x': x_small,
    'actual_y': y_small,
    'predicted_y': pred_small,
    'error = predicted - actual': errors_small,
    'squared_error': errors_small ** 2,
    'error*x': errors_small * x_small,
}).round(3)

display(small_df)
print(f'MSE = mean(squared_error) = {mse_small:.3f}')
print(f'Gradient for w = mean(error * x) = {grad_small:.3f}')
print()
print('Interpretation:')
print('- MSE tells us the current weight w=1.0 gives a poor fit.')
print('- The negative gradient tells us to increase the weight.')
print('- If we increase the weight, predictions move closer to the true values 4, 6, 8.')

## Section 3 — How the Gradient Blames Each Feature for the Error

### The prediction error

For each observation:
```
error = predicted_spend − actual_spend
```
Positive error → we over-predicted (too high)
Negative error → we under-predicted (too low)

### How much did each weight contribute to the error?

The gradient tells us: if we increase weight `w_age` by a tiny amount, how much does the error change?

```
gradient of w_age = (1/N) × Σ  error × age_value
```

**Why multiply error by the feature value?**

Think about it:
- If `age` was 50 for an observation where we over-predicted by $100:
  `gradient = 100 × 50 = 5000` — the age weight was very involved (age was large)
- If `age` was 2 for the same error:
  `gradient = 100 × 2 = 200` — the age weight had little involvement (age was small)

The feature value is like a **lever**:
- A large feature value amplifies the weight's contribution to the error
- A near-zero feature value means that weight barely affected the prediction for that observation

### The update rule

```
new_weight = old_weight − learning_rate × gradient
```

- Gradient is positive → prediction was too high → reduce weight
- Gradient is negative → prediction was too low → increase weight

In [ ]:
# Show one gradient step in detail on a 5-observation sample
sample_idx = [0, 10, 50, 100, 200]
X_sample = X[sample_idx]
y_sample  = y[sample_idx]

y_hat  = X_sample @ w_random + b_random
errors = y_hat - y_sample

print("One gradient step — 5 observations, 4 features")
print("=" * 70)
print(f"{'Obs':>4}  {'age':>7}  {'income':>10}  {'score':>7}  {'is_new':>7}  {'error':>8}")
print("-" * 70)
for i in range(len(sample_idx)):
    print(f"{sample_idx[i]:>4}  "
          f"{X_sample[i,0]:>7.1f}  "
          f"{X_sample[i,1]:>10.0f}  "
          f"{X_sample[i,2]:>7.1f}  "
          f"{X_sample[i,3]:>7.0f}  "
          f"{errors[i]:>8.1f}")

print()
gradients = (X_sample.T @ errors) / len(sample_idx)
print("Gradient for each weight = mean(error × feature_value):")
for feat, g in zip(FEATURES, gradients):
    direction = 'decrease weight ↓' if g > 0 else 'increase weight ↑'
    print(f"  grad_{feat:8s} = {g:>10.2f}   → {direction}")
print()
print("Notice: income gradient is huge because income values are huge (thousands).")
print("This is the scale problem that StandardScaler solves.")

### Student Walk-Through: Why `error × feature` updates the weight

Before using many observations and many features, it helps to slow the idea down.

Suppose the model has only **one feature**:

```python
prediction = w × x
```

We train it on three rows:

| Row | x | y |
|---|---:|---:|
| 1 | 2 | 4 |
| 2 | 3 | 6 |
| 3 | 4 | 8 |

Start with:
- `w = 1.0`
- learning rate = `0.1`

For each row, training does four things:
1. predict with the current weight
2. compute `error = prediction − actual`
3. compute `gradient = error × feature`
4. update `w = w − learning_rate × gradient`

This is the key idea:
- `error` says **how wrong** the model was
- `feature` says **how much this input participated** in the prediction
- `error × feature` says **how much to correct that weight**

If the prediction is too low, the error is negative, so the weight increases.
If the prediction is too high, the error is positive, so the weight decreases.

In [ ]:
# Slow-motion walk-through on a 1-feature dataset
x_demo = np.array([2.0, 3.0, 4.0])
y_demo = np.array([4.0, 6.0, 8.0])

w = 1.0
lr = 0.1
rows = []

for step, (x_i, y_i) in enumerate(zip(x_demo, y_demo), start=1):
    y_hat = w * x_i
    error = y_hat - y_i
    gradient = error * x_i
    new_w = w - lr * gradient
    rows.append({
        'step': step,
        'x': x_i,
        'y': y_i,
        'prediction': round(y_hat, 3),
        'error': round(error, 3),
        'gradient = error*x': round(gradient, 3),
        'new_weight': round(new_w, 3)
    })
    print(f"Step {step}: x={x_i:.0f}, y={y_i:.0f}, prediction={y_hat:.2f}, error={error:.2f}, gradient={gradient:.2f}, new_weight={new_w:.3f}")
    w = new_w

print()
walkthrough_df = pd.DataFrame(rows)
display(walkthrough_df)

print('What students should notice:')
print('- Row 1 under-predicts, so error is negative and the weight increases.')
print('- Row 2 still under-predicts, so the weight increases again.')
print('- As the weight gets closer to the correct value, the error becomes smaller.')
print('- Smaller error produces a smaller gradient and therefore a smaller update.')
print(f'- After 3 rows, the weight has moved from 1.0 to {w:.3f}, close to the ideal value 2.0.')

## Section 4 — Does the Feature's Distribution Determine Its Weight?

### The answer: No — and here is why

The weight is determined by the **relationship between the feature and the target**,
not by the feature's distribution.

But the **distribution affects three things**:
1. The **scale** of the raw weight
2. The **speed** at which the weight is learned
3. The **comparability** of weights across features

### Demonstration: same real-world effect, very different raw weights

In our dataset:
- `age` (range ~18–70): true weight = **0.8** — each extra year of age → $0.80 more spent
- `income` (range ~$20k–$300k): true weight = **0.0003** — each extra dollar → $0.0003 more spent

But their *total contribution* across a typical range is similar:
- age effect across its range: 0.8 × (70−18) = **$41.6**
- income effect across its range: 0.0003 × (300k−20k) = **$84.0**

The weights look completely different (0.8 vs 0.0003) but they represent similar-scale effects
on the target. **The weight encodes the effect per ONE UNIT of the feature.**
Since the units are different (years vs dollars), the raw weights are not comparable.

### After StandardScaler: weights become comparable

Once we scale both features to mean=0, std=1, one "unit" means one standard deviation.
Now the scaled weight directly answers: "how much does the target change per
standard deviation of this feature?" — and the weights are directly comparable.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Show what the true weights become in scaled space
# If raw weight for feature i is w_i and scaler std is s_i:
# scaled_weight_i = w_i × s_i  (because scaled_x = (x - mean) / std)
print("Feature standard deviations (raw):")
for feat, std in zip(FEATURES, X.std(axis=0)):
    print(f"  {feat:8s}: std = {std:>10.2f}")

print()
print("True weights — raw vs scaled comparison:")
print(f"{'Feature':10s}  {'Raw weight':>12}  {'Std':>10}  {'Scaled weight':>14}  {'Interpretation'}")
print("-" * 75)
for feat, w_raw, std in zip(FEATURES, w_true, X.std(axis=0)):
    w_scaled = w_raw * std
    print(f"{feat:10s}  {w_raw:>12.4f}  {std:>10.2f}  {w_scaled:>14.2f}  "
          f"${ abs(w_scaled):.1f} per 1-std {'increase' if w_scaled>0 else 'decrease'}")

print()
print("After scaling, the magnitudes are comparable — income and age have similar effect sizes.")
print("The raw weights (0.8 and 0.0003) looked very different but the underlying effect is similar.")

In [ ]:
# Scatter: each feature vs target — the slope IS the true weight
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['steelblue', '#FFA726', '#26A69A', '#AB47BC']

for ax, (feat, color) in zip(axes, zip(FEATURES, colors)):
    x_vals = df[feat].values
    ax.scatter(x_vals, y, alpha=0.15, s=8, color=color)
    # Fit a simple line to show the marginal slope
    from numpy.polynomial.polynomial import polyfit
    b_fit, m_fit = polyfit(x_vals, y, 1)
    xr = np.array([x_vals.min(), x_vals.max()])
    ax.plot(xr, m_fit * xr + b_fit, 'red', lw=2,
            label=f'slope ≈ {m_fit:.4f}')
    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel('spend ($)', fontsize=10)
    ax.set_title(f'{feat}\nTrue weight = {TRUE_W[feat]}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Each Feature vs Target — the Slope Shows the Relationship (not the distribution)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("The scatter plot shape (normal/skewed/binary) does not determine the slope.")
print("The slope = the weight. It measures how the target changes per unit of the feature.")

## Section 5 — Gradient Descent in Slow Motion: Watch Weights Converge

### We now run gradient descent from scratch

Starting from random weights, we apply the update rule thousands of times:

```python
for each iteration:
    y_hat    = X @ w + b               # predict
    error    = y_hat - y               # measure error
    grad_w   = (1/N) × X.T @ error     # gradient for each weight
    grad_b   = (1/N) × error.sum()     # gradient for bias
    w        = w - learning_rate × grad_w   # update weights
    b        = b - learning_rate × grad_b   # update bias
```

We run this **twice**: once on raw (unscaled) features, once on scaled features.
The comparison reveals why scaling is critical.

In [ ]:
def gradient_descent(X_data, y_data, lr, n_iter, true_weights=None):
    n, p = X_data.shape
    w = np.zeros(p)
    b = 0.0
    mse_history = []
    w_history   = [w.copy()]

    for _ in range(n_iter):
        y_hat  = X_data @ w + b
        error  = y_hat - y_data
        grad_w = X_data.T @ error / n
        grad_b = error.mean()
        w = w - lr * grad_w
        b = b - lr * grad_b
        mse_history.append(np.mean(error**2))
        w_history.append(w.copy())

    return w, b, np.array(mse_history), np.array(w_history)

# ── Run on raw features ───────────────────────────────────────────────────────
w_raw, b_raw, mse_raw, wh_raw = gradient_descent(X, y, lr=1e-8, n_iter=2000)

# ── Run on scaled features ────────────────────────────────────────────────────
w_sc, b_sc, mse_sc, wh_sc = gradient_descent(X_scaled, y, lr=0.05, n_iter=2000)

# Convert scaled weights back to raw space for comparison
w_sc_raw_space = w_sc / X.std(axis=0)

print("After 2000 iterations:")
print()
print(f"{'Feature':10s}  {'True':>10}  {'Learned (raw X)':>16}  {'Learned (scaled X)':>20}")
print("-" * 62)
for i, feat in enumerate(FEATURES):
    print(f"{feat:10s}  {w_true[i]:>10.4f}  {w_raw[i]:>16.4f}  {w_sc_raw_space[i]:>20.4f}")
print()
print(f"Final MSE — raw features  : {mse_raw[-1]:>10,.1f}")
print(f"Final MSE — scaled features: {mse_sc[-1]:>10,.1f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ── Top row: MSE convergence ──────────────────────────────────────────────────
axes[0, 0].semilogy(mse_raw, color='#EF5350', lw=1.5)
axes[0, 0].axhline(mse_sc[-1], color='#26A69A', ls='--', lw=1.5, label='Scaled floor')
axes[0, 0].set_title('MSE During Training — Raw Features\n(log scale)', fontsize=11)
axes[0, 0].set_xlabel('Iteration', fontsize=10)
axes[0, 0].set_ylabel('MSE (log scale)', fontsize=10)
axes[0, 0].legend()

axes[0, 1].semilogy(mse_sc, color='#26A69A', lw=1.5)
axes[0, 1].set_title('MSE During Training — Scaled Features\n(log scale)', fontsize=11)
axes[0, 1].set_xlabel('Iteration', fontsize=10)
axes[0, 1].set_ylabel('MSE (log scale)', fontsize=10)

# ── Bottom row: weight convergence ────────────────────────────────────────────
feat_colors = ['steelblue', '#FFA726', '#26A69A', '#AB47BC']

for i, (feat, color) in enumerate(zip(FEATURES, feat_colors)):
    true_val = w_true[i]
    axes[1, 0].plot(wh_raw[:, i], color=color, lw=1.5, alpha=0.8, label=feat)
    axes[1, 0].axhline(true_val, color=color, ls=':', lw=1, alpha=0.5)

axes[1, 0].set_title('Raw Features: Weight Paths\n(dotted line = true weight)', fontsize=11)
axes[1, 0].set_xlabel('Iteration', fontsize=10)
axes[1, 0].set_ylabel('Weight value', fontsize=10)
axes[1, 0].legend(fontsize=9)

# Scaled weights in raw space
true_scaled = w_true * X.std(axis=0)   # true weights in scaled space
for i, (feat, color) in enumerate(zip(FEATURES, feat_colors)):
    axes[1, 1].plot(wh_sc[:, i], color=color, lw=1.5, alpha=0.8, label=feat)
    axes[1, 1].axhline(true_scaled[i], color=color, ls=':', lw=1, alpha=0.5)

axes[1, 1].set_title('Scaled Features: Weight Paths\n(dotted line = true weight in scaled space)', fontsize=11)
axes[1, 1].set_xlabel('Iteration', fontsize=10)
axes[1, 1].set_ylabel('Weight value (scaled space)', fontsize=10)
axes[1, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Section 6 — Why Feature Scale Controls Learning Speed

### Look at what happens to each feature's gradient

The gradient update for weight `w_i` is:
```
gradient_i = (1/N) × Σ  error × x_i_value
```

For `income`, values are in the **tens of thousands** → gradients are enormous.
For `is_new`, values are **0 or 1** → gradients are tiny.

When we apply the same learning rate to both:
- `income` weight overshoots wildly → instability
- `is_new` weight barely moves → very slow learning

### This is not a flaw in the algorithm — it is a units problem

The algorithm treats every feature equally.
It does not know that 1 income unit = $1 and 1 age unit = 1 year.
The scale difference is purely in the data.

StandardScaler removes this problem by making every feature's 1 unit = 1 standard deviation.
Now the gradient for every feature is on the same scale, and any single learning rate works well.

### The weight is not learned faster because of the distribution shape

Two features could both be normally distributed but have different scales.
The learning speed depends on the **scale** (range / variance), not the shape (normal, skewed, etc.).

In [ ]:
# Demonstrate: same distribution shape, different scale → very different gradient magnitudes
np.random.seed(42)
x_small = np.random.normal(0, 1,   N)   # std = 1
x_large = np.random.normal(0, 500, N)   # std = 500  (same shape, 500x larger)

# Both have the same true relationship with a toy target
y_toy = 2.0 * x_small + 0.004 * x_large + np.random.normal(0, 1, N)
# true effect: 2 × std(1) = 2.0   and  0.004 × std(500) = 2.0  → same real-world contribution

X_toy = np.column_stack([x_small, x_large])
w_toy, b_toy, mse_toy, wh_toy = gradient_descent(X_toy, y_toy, lr=1e-6, n_iter=3000)

X_toy_sc = StandardScaler().fit_transform(X_toy)
w_toy_sc, _, mse_toy_sc, wh_toy_sc = gradient_descent(X_toy_sc, y_toy, lr=0.1, n_iter=3000)

print("Two features: same distribution SHAPE (Normal), different SCALE")
print()
print(f"x_small: std={x_small.std():.1f}  true weight = 2.0")
print(f"x_large: std={x_large.std():.1f}  true weight = 0.004")
print()
print(f"Both have the same real-world contribution: weight × std = 2.0")
print()
print(f"After 3000 iterations (raw):")
print(f"  x_small weight: {w_toy[0]:.4f}  (true: 2.0000)")
print(f"  x_large weight: {w_toy[1]:.6f}  (true: 0.0040)")
print()
w_toy_sc_raw = w_toy_sc / X_toy.std(axis=0)
print(f"After 3000 iterations (scaled):")
print(f"  x_small weight: {w_toy_sc_raw[0]:.4f}  (true: 2.0000)")
print(f"  x_large weight: {w_toy_sc_raw[1]:.6f}  (true: 0.0040)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(wh_toy[:, 0], color='steelblue',  lw=2, label='x_small (std=1)')
axes[0].plot(wh_toy[:, 1], color='orangered', lw=2, label='x_large (std=500)')
axes[0].axhline(2.0,   color='steelblue',  ls='--', lw=1, alpha=0.6, label='True: 2.0')
axes[0].axhline(0.004, color='orangered', ls='--', lw=1, alpha=0.6, label='True: 0.004')
axes[0].set_title('Raw Features — Unequal learning rates per feature\n(same shape, different scale)', fontsize=11)
axes[0].set_xlabel('Iteration', fontsize=10)
axes[0].set_ylabel('Weight value', fontsize=10)
axes[0].legend(fontsize=9)

# Scaled weights: both converge smoothly
true_sc = np.array([2.0, 0.004]) * X_toy.std(axis=0)
axes[1].plot(wh_toy_sc[:, 0], color='steelblue',  lw=2, label='x_small (scaled)')
axes[1].plot(wh_toy_sc[:, 1], color='orangered', lw=2, label='x_large (scaled)')
axes[1].axhline(true_sc[0], color='steelblue',  ls='--', lw=1, alpha=0.6)
axes[1].axhline(true_sc[1], color='orangered', ls='--', lw=1, alpha=0.6)
axes[1].set_title('Scaled Features — Both weights converge at the same rate\n(shape does not matter)', fontsize=11)
axes[1].set_xlabel('Iteration', fontsize=10)
axes[1].set_ylabel('Weight value (scaled space)', fontsize=10)
axes[1].legend(fontsize=9)

plt.suptitle('Distribution Shape Does NOT Control Learning Speed — Scale Does', fontsize=12)
plt.tight_layout()
plt.show()

## Section 7 — Reading the Final Weights: What Do They Tell Us?

### Raw weights: effect per one unit of the feature (depends on units)

| Feature | Raw weight | Interpretation |
|---|---|---|
| `age` (years) | 0.8 | +$0.80 per additional year of age |
| `income` ($) | 0.0003 | +$0.03 per additional $100 income |
| `score` (0–100) | 1.5 | +$1.50 per additional point |
| `is_new` (0/1) | −30.0 | new customers spend $30 less |

Raw weights are hard to compare because the features are in different units.

### Scaled weights: effect per one standard deviation (comparable)

After StandardScaler, the weight tells you:
"how much does the target change when this feature increases by one standard deviation?"

Now you can directly compare: which feature has the **biggest impact** on the target?

### Summary of concepts learned

In [ ]:
from sklearn.linear_model import LinearRegression

# Train sklearn's LinearRegression on scaled features
X_sc_final = StandardScaler().fit_transform(X)
lr_model   = LinearRegression().fit(X_sc_final, y)

summary = pd.DataFrame({
    'Feature'          : FEATURES,
    'Distribution'     : ['Normal', 'Log-normal', 'Uniform', 'Binary'],
    'Raw true weight'  : [TRUE_W[f] for f in FEATURES],
    'Std of feature'   : X.std(axis=0).round(2),
    'Scaled weight'    : (lr_model.coef_).round(3),
    'Scaled |weight|'  : np.abs(lr_model.coef_).round(3),
})
summary = summary.sort_values('Scaled |weight|', ascending=False)

print("=== Final Weight Summary ===")
print(summary.drop(columns='Scaled |weight|').to_string(index=False))
print()
print("The distribution column has NO effect on the weight values.")
print("The weight is purely determined by the relationship with the target.")
print()
print("Reading the scaled weights:")
for _, row in summary.iterrows():
    direction = 'increases' if row['Scaled weight'] > 0 else 'decreases'
    print(f"  {row['Feature']:8s} ({row['Distribution']:10s}): "
          f"1 std {direction} spend by ${abs(row['Scaled weight']):.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw weights (hard to compare)
feat_colors = ['steelblue', '#FFA726', '#26A69A', '#AB47BC']
raw_weights = [TRUE_W[f] for f in FEATURES]
colors_raw  = ['#26A69A' if w >= 0 else '#EF5350' for w in raw_weights]
ax1.barh(FEATURES, raw_weights, color=colors_raw)
ax1.axvline(0, color='black', lw=0.8)
ax1.set_title('Raw Weights (per 1 unit of feature)\nNot directly comparable — different units', fontsize=11)
ax1.set_xlabel('Weight value', fontsize=10)

# Right: scaled weights (comparable)
scaled_w   = lr_model.coef_
colors_sc  = ['#26A69A' if w >= 0 else '#EF5350' for w in scaled_w]
ax2.barh(FEATURES, scaled_w, color=colors_sc)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_title('Scaled Weights (per 1 std of feature)\nDirectly comparable — which feature matters most?', fontsize=11)
ax2.set_xlabel('Weight value (scaled space)', fontsize=10)

plt.suptitle('Raw vs Scaled Weights — Same Model, Different Perspective', fontsize=12)
plt.tight_layout()
plt.show()
print()
print('is_new has the largest scaled weight — being a new customer is the strongest predictor.')
print('income has the smallest scaled weight despite looking important in raw form.')

## Summary — Key Concepts

### 1. What is an observation, a feature, a target?
- **Observation** = one row = one real-world instance measured
- **Feature** = one column = one measurement taken per observation
- **Target** = the column we want to predict

### 2. What is a weight?
- A number assigned to each feature
- It says: "for every 1 unit increase in this feature, the prediction changes by *weight* units"
- The model's prediction = weighted sum of all features + bias

### 3. How is the weight learned?
- Start from random (or zero) weights
- Predict → measure error → compute gradient (error × feature value) → update weight
- Repeat thousands of times until error is minimised (gradient descent)

### 4. Does the feature's distribution determine its weight?

**No.** The weight is determined by the **relationship between feature and target** (the slope).
The distribution shape (normal, skewed, binary, uniform) has no direct effect on the weight.

However, the distribution **does** affect:
- The **scale** of the raw weight — features in larger units have smaller weights
- The **gradient size** — large-scale features produce large gradients → dominate gradient descent
- The **convergence stability** — without scaling, some weights overshoot, others barely move

### 5. Why does StandardScaler help?
- It puts all features on the same scale (mean=0, std=1)
- All gradients are now on the same scale → learning rate works uniformly for all features
- Scaled weights are directly comparable: larger |weight| = stronger predictor
- The model converges faster and more stably

### 6. What does the final weight tell you?
- **Raw weight**: change in target per 1 unit of the feature (in the feature's own units)
- **Scaled weight**: change in target per 1 standard deviation (comparable across all features)
- **Sign**: direction (positive = increases target, negative = decreases target)
- **Magnitude**: strength (larger = more influential feature)